# 2. Text Splitters (Chia nhỏ văn bản)

## 📖 Lý thuyết
Khi đưa tài liệu vào mô hình AI (LLM), chúng ta bị giới hạn bởi **Context Window** (số lượng từ tối đa mô hình có thể đọc cùng lúc). Hơn nữa, tìm kiếm trên văn bản quá dài sẽ không chính xác.

Do đó, chúng ta cần **Chunking** - chia tài liệu lớn thành các đoạn nhỏ (`chunk`).
Trong LangChain, công cụ phổ biến nhất là `RecursiveCharacterTextSplitter`. Nó cố gắng chia theo đoạn văn (`\n\n`), rồi đến câu (`\n`), rồi đến từ (` `) để giữ trọn vẹn ngữ nghĩa.
- `chunk_size`: Kích thước tối đa của một chunk.
- `chunk_overlap`: Kích thước phần giao nhau (overlap) giữa 2 chunk liền kề để không làm đứt đoạn ngữ cảnh.

## 💻 Code mẫu
Chúng ta sẽ mock một đoạn text dài và dùng `RecursiveCharacterTextSplitter` với thông số giống `chunking.py` trong dự án RAG SGU.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# 1. Tạo một Document mẫu
sample_text = (
    "Chương 1: Quy định chung.\n\n" + 
    "Điều 1. Phạm vi điều chỉnh.\n" +
    "Quy chế này quy định về đào tạo trình độ đại học hình thức chính quy.\n\n" +
    "Điều 2. Đối tượng áp dụng.\n" +
    "Quy chế này áp dụng đối với sinh viên, giảng viên và các phòng ban liên quan.\n\n"
    "(Đây là một đoạn text rất rất dài... ) " * 20
)
doc = Document(page_content=sample_text, metadata={"source": "File_PDFs/Bản sao của BanMOTa_CNTT_2020-2024.pdf", "page": 1})

# 2. Khởi tạo Splitter (Sử dụng config giống dự án: size=1600, overlap=200)
chunk_size = 200  # Đặt mức nhỏ để dễ demo
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", " ", ""], # Ưu tiên cắt theo đoạn trước
    length_function=len
)

# 3. Tiến hành chia nhỏ
chunks = splitter.split_documents([doc])
print(f"Đã chia thành {len(chunks)} chunks.\n")

for i, chunk in enumerate(chunks[:3]): # In thử 3 chunk đầu
    print(f"--- Chunk {i+1} (Dài {len(chunk.page_content)} ký tự) ---")
    print(chunk.page_content)
    print()


Đã chia thành 21 chunks.

--- Chunk 1 (Dài 124 ký tự) ---
Chương 1: Quy định chung.

Điều 1. Phạm vi điều chỉnh.
Quy chế này quy định về đào tạo trình độ đại học hình thức chính quy.

--- Chunk 2 (Dài 104 ký tự) ---
Điều 2. Đối tượng áp dụng.
Quy chế này áp dụng đối với sinh viên, giảng viên và các phòng ban liên quan.

--- Chunk 3 (Dài 116 ký tự) ---
(Đây là một đoạn text rất rất dài... ) Quy chế này áp dụng đối với sinh viên, giảng viên và các phòng ban liên quan.



## 🎯 Bài tập nhỏ
**Yêu cầu**: Thay đổi `chunk_size=50` và `chunk_overlap=10`. Chạy lại hàm split và quan sát sự thay đổi của số lượng chunk và nội dung bên trong mỗi chunk bị cắt vụn như thế nào.

In [5]:
# Khởi tạo Splitter với kích thước nhỏ hơn
small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    separators=["\n\n", "\n", " ", ""],
    length_function=len
)

small_chunks = small_splitter.split_documents([doc])
print(f"Đã chia thành {len(small_chunks)} chunks (nhiều hơn rất nhiều so với chunk_size=200).\n")

for i, chunk in enumerate(small_chunks[:5]): # Xem 5 chunk đầu tiên
    print(f"--- Chunk {i+1} (Dài {len(chunk.page_content)} ký tự) ---")
    print(chunk.page_content)
    


Đã chia thành 65 chunks (nhiều hơn rất nhiều so với chunk_size=200).

--- Chunk 1 (Dài 25 ký tự) ---
Chương 1: Quy định chung.
--- Chunk 2 (Dài 27 ký tự) ---
Điều 1. Phạm vi điều chỉnh.
--- Chunk 3 (Dài 48 ký tự) ---
Quy chế này quy định về đào tạo trình độ đại học
--- Chunk 4 (Dài 28 ký tự) ---
đại học hình thức chính quy.
--- Chunk 5 (Dài 26 ký tự) ---
Điều 2. Đối tượng áp dụng.
